## Install dependencies


In [ ]:
# Run once
# !pip install statsbombpy requests beautifulsoup4 fuzzywuzzy python-Levenshtein lxml pandas numpy soccerdata

## Imports


In [2]:
import pandas as pd
import numpy as np
import requests
import time
import warnings
warnings.filterwarnings('ignore')
from bs4 import BeautifulSoup
from fuzzywuzzy import process, fuzz
from statsbombpy import sb

## STEP 1 — Get all unique players from La Liga data


In [3]:
COMP_ID, SEASON_ID = 11, 90
matches = sb.matches(competition_id=COMP_ID, season_id=SEASON_ID)

all_events = []
for mid in matches.match_id:
    ev = sb.events(match_id=mid)
    ev['match_id'] = mid
    all_events.append(ev[ev.type == 'Pass'][['player', 'player_id']].drop_duplicates())

players_df = (pd.concat(all_events)
              .drop_duplicates('player_id')
              .reset_index(drop=True)
              .sort_values('player'))

print(f"Unique players to look up: {len(players_df)}")
print(players_df.head(10).to_string(index=False))

Unique players to look up: 379
                    player  player_id
Aarón Escandell Banacloche    25704.0
      Aarón Martín Caricol     6757.0
             Adnan Januzaj     6330.0
      Adrián López Álvarez     6735.0
        Adrián Marín Gómez     6935.0
Aitor Fernández Abarisketa    23344.0
       Aitor Ruibal García     6851.0
      Alberto Moreno Pérez     3515.0
    Alberto Perea Correoso    23811.0
    Alberto Rodríguez Baró    30419.0


## APPROACH A — FBref via soccerdata (most citable)


In [4]:
# pip install soccerdata
# This wraps FBref scraping with caching + rate limiting built in
try:
    import soccerdata as sd

    fbref = sd.FBref(leagues='ESP-La Liga', seasons='2020-2021')
    bio   = fbref.read_player_season_stats(stat_type='standard')
    print("Columns:", bio.columns.tolist())

    if 'foot' in bio.columns:
        foot_fbref = (bio[['player', 'foot']]
                      .drop_duplicates('player')
                      .rename(columns={'foot': 'preferred_foot'}))
        print(f"\nFBref: {len(foot_fbref)} players with foot data")
        print(foot_fbref['preferred_foot'].value_counts())
        # Save
        foot_fbref.to_csv('preferred_foot_fbref.csv', index=False)
    else:
        print("'foot' column not found — check bio.columns above")

except ImportError:
    print("Run: pip install soccerdata")
except Exception as e:
    print(f"Error: {e}")

[07/06/26 16:13:16] INFO     No custom team name replacements found. You can configure these in       ]8;id=4414799;file:///home/ankit/anaconda3/envs/xpass/lib/python3.14/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=4414800;file:///home/ankit/anaconda3/envs/xpass/lib/python3.14/site-packages/soccerdata/_config.py#92\92]8;;\
                             /home/ankit/soccerdata/config/teamname_replacements.json.                             

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=4414806;file:///home/ankit/anaconda3/envs/xpass/lib/python3.14/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=4414807;file:///home/ankit/anaconda3/envs/xpass/lib/python3.14/site-packages/soccerdata/_config.py#190\190]8;;\
                             /home/ankit/soccerdata/config/league_dict.json.                                       

                    INFO     Saving cached data to /home/ankit/soccerdata/data/FBref                 ]8;id=4414814;file:///home/ankit/anaconda3/envs/xpass/lib/python3.14/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=4414815;file:///home/ankit/anaconda3/envs/xpass/lib/python3.14/site-packages/soccerdata/_common.py#250\250]8;;\

Columns: [('nation', ''), ('pos', ''), ('age', ''), ('born', ''), ('Playing Time', 'MP'), ('Playing Time', 'Starts'), ('Playing Time', 'Min'), ('Playing Time', '90s'), ('Performance', 'Gls'), ('Performance', 'Ast'), ('Performance', 'G+A'), ('Performance', 'G-PK'), ('Performance', 'PK'), ('Performance', 'PKatt'), ('Performance', 'CrdY'), ('Performance', 'CrdR'), ('Per 90 Minutes', 'Gls'), ('Per 90 Minutes', 'Ast'), ('Per 90 Minutes', 'G+A'), ('Per 90 Minutes', 'G-PK'), ('Per 90 Minutes', 'G+A-PK')]
'foot' column not found — check bio.columns above


## APPROACH B — Transfermarkt scraper (most complete)


In [ ]:
def get_foot_transfermarkt(player_name, session):
    """Search Transfermarkt for a player and return their preferred foot."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept': 'text/html,application/xhtml+xml',
    }
    search_url = "https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche"
    try:
        r = session.get(search_url,
                        params={'query': player_name, 'Spieler_page': '0'},
                        headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, 'lxml')

        # Get first player result link
        result = soup.select_one('table.items td.hauptlink a')
        if not result:
            return None

        player_url = "https://www.transfermarkt.com" + result['href']
        time.sleep(0.8)  # be polite to the server

        r2   = session.get(player_url, headers=headers, timeout=10)
        soup2 = BeautifulSoup(r2.text, 'lxml')

        # Foot appears in the player profile info table
        for li in soup2.select('.info-table__content'):
            txt = li.get_text(strip=True)
            if txt in ('Right', 'Left', 'Both feet', 'both'):
                return txt.replace(' feet', '').title()

        for span in soup2.find_all('span', class_='info-table__content--bold'):
            txt = span.get_text(strip=True)
            if txt in ('Right', 'Left', 'Both feet'):
                return txt.replace(' feet', '').title()

    except Exception as e:
        print(f"  Error for {player_name}: {e}")
    return None


def scrape_transfermarkt(players_df, n_players=None):
    """Scrape preferred foot from Transfermarkt. n_players=None scrapes all."""
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})

    results = []
    subset  = players_df.head(n_players) if n_players else players_df

    for i, (_, row) in enumerate(subset.iterrows()):
        name = row['player']
        print(f"  [{i+1}/{len(subset)}] {name}...", end=' ', flush=True)
        foot = get_foot_transfermarkt(name, session)
        print(foot or "not found")
        results.append({'player_id': row['player_id'], 'player': name,
                        'preferred_foot': foot})
        time.sleep(0.5)

    df = pd.DataFrame(results)
    print(f"\nFound foot data for {df.preferred_foot.notna().sum()}/{len(df)} players")
    print(df['preferred_foot'].value_counts(dropna=False))
    return df

# ── Run this (takes ~3-4 min for full squad) ──
# foot_tm = scrape_transfermarkt(players_df)
# foot_tm.to_csv('preferred_foot_transfermarkt.csv', index=False)

# ── Quick test with 5 players first ──
# foot_tm_test = scrape_transfermarkt(players_df, n_players=5)

## APPROACH C — FIFA 21 dataset (fastest — download from Kaggle)


In [ ]:
# Download from: https://www.kaggle.com/datasets/stefanoleone992/fifa-21-complete-player-dataset
# File: players_21.csv  (~45MB)
# No scraping needed — just download and point to the path below

def load_fifa_foot(fifa_csv_path='players_21.csv'):
    """Load preferred foot from FIFA 21 dataset."""
    fifa = pd.read_csv(fifa_csv_path,
                       usecols=['short_name', 'long_name', 'preferred_foot'],
                       low_memory=False)
    fifa = fifa.drop_duplicates('long_name').copy()
    print(f"FIFA 21: {len(fifa)} players loaded")
    print(f"Foot distribution:\n{fifa['preferred_foot'].value_counts()}")
    return fifa


def match_fifa_to_statsbomb(statsbomb_players, fifa_df, threshold=85):
    """Fuzzy name match StatsBomb players to FIFA dataset."""
    fifa_names = fifa_df['long_name'].tolist()
    matched    = []

    for _, row in statsbomb_players.iterrows():
        sb_name = row['player']

        # 1. Exact match
        exact = fifa_df[fifa_df['long_name'].str.lower() == sb_name.lower()]
        if len(exact) > 0:
            foot = exact.iloc[0]['preferred_foot']
            matched.append({'player_id': row['player_id'], 'player': sb_name,
                            'preferred_foot': foot, 'match_score': 100,
                            'matched_name': exact.iloc[0]['long_name']})
            continue

        # 2. Fuzzy match
        best_match, score = process.extractOne(
            sb_name, fifa_names, scorer=fuzz.token_sort_ratio)
        if score >= threshold:
            foot = fifa_df[fifa_df['long_name'] == best_match].iloc[0]['preferred_foot']
            matched.append({'player_id': row['player_id'], 'player': sb_name,
                            'preferred_foot': foot, 'match_score': score,
                            'matched_name': best_match})
        else:
            matched.append({'player_id': row['player_id'], 'player': sb_name,
                            'preferred_foot': None, 'match_score': score,
                            'matched_name': best_match})

    df = pd.DataFrame(matched)
    matched_n = df['preferred_foot'].notna().sum()
    print(f"\nMatched: {matched_n}/{len(df)} players ({matched_n/len(df):.1%})")
    print(f"Unmatched players: {df[df.preferred_foot.isna()]['player'].tolist()}")
    print(f"\nLow-confidence matches (score < 90):")
    low = df[(df.match_score < 90) & df.preferred_foot.notna()]
    print(low[['player','matched_name','match_score','preferred_foot']].to_string(index=False))
    return df

# ── Run after downloading players_21.csv ──
# fifa_df = load_fifa_foot('players_21.csv')
# foot_df = match_fifa_to_statsbomb(players_df, fifa_df)
# foot_df.to_csv('preferred_foot_fifa.csv', index=False)

## APPROACH D — Wikidata SPARQL (free, open, citable)


In [ ]:
def fetch_wikidata_foot():
    """
    Fetch preferred foot for all footballers from Wikidata public SPARQL.
    Wikidata property P2354 = preferred foot.
    Completely free, open, and academically citable.
    """
    endpoint = "https://query.wikidata.org/sparql"
    query = """
    SELECT DISTINCT ?playerLabel ?foot ?footLabel WHERE {
      ?player wdt:P106 wd:Q937857 .   # occupation: association football player
      ?player wdt:P2354 ?foot .        # preferred foot
      SERVICE wikibase:label {
        bd:serviceParam wikibase:language "en" .
      }
    }
    """
    r = requests.get(
        endpoint,
        params={'query': query, 'format': 'json'},
        headers={
            'User-Agent': 'xPassResearch/1.0 (academic research)',
            'Accept':     'application/sparql-results+json',
        },
        timeout=60
    )
    if r.status_code != 200:
        print(f"Wikidata error {r.status_code}: {r.text[:200]}")
        return pd.DataFrame()

    results = r.json()['results']['bindings']
    rows = [
        {'player_wiki':    p['playerLabel']['value'],
         'preferred_foot': p['footLabel']['value'].replace(' foot', '').title()}
        for p in results
    ]
    df = pd.DataFrame(rows).drop_duplicates('player_wiki')
    print(f"Wikidata: {len(df)} footballers with preferred foot")
    print(df['preferred_foot'].value_counts())
    return df


def match_wikidata_to_statsbomb(statsbomb_players, wikidata_df, threshold=85):
    """Fuzzy match Wikidata player names to StatsBomb names."""
    wiki_names = wikidata_df['player_wiki'].tolist()
    matched    = []

    for _, row in statsbomb_players.iterrows():
        sb_name     = row['player']
        best, score = process.extractOne(sb_name, wiki_names, scorer=fuzz.token_sort_ratio)
        if score >= threshold:
            foot = wikidata_df[wikidata_df['player_wiki'] == best].iloc[0]['preferred_foot']
            matched.append({'player_id': row['player_id'], 'player': sb_name,
                            'preferred_foot': foot, 'score': score, 'matched': best})
        else:
            matched.append({'player_id': row['player_id'], 'player': sb_name,
                            'preferred_foot': None, 'score': score, 'matched': best})

    df = pd.DataFrame(matched)
    n  = df.preferred_foot.notna().sum()
    print(f"Matched: {n}/{len(df)} players ({n/len(df):.1%})")
    return df

# ── Run ──
# wiki_df   = fetch_wikidata_foot()
# foot_wiki = match_wikidata_to_statsbomb(players_df, wiki_df)
# foot_wiki.to_csv('preferred_foot_wikidata.csv', index=False)

## STEP 2 — Validate: inferred vs ground-truth preferred foot


In [ ]:
def validate_inference(passes_with_inferred, foot_ground_truth_df):
    """
    Compare your usage-inferred preferred foot against ground truth.
    The accuracy number is itself a finding worth reporting.
    """
    # Inferred: from Cell 4 in your main notebook (pass usage distribution)
    inferred = (passes_with_inferred[['player_id', 'player', 'preferred_foot']]
                .drop_duplicates('player_id')
                .rename(columns={'preferred_foot': 'inferred'}))

    # Ground truth: from whichever approach above returned data
    ground   = (foot_ground_truth_df[['player_id', 'preferred_foot']]
                .rename(columns={'preferred_foot': 'ground_truth'}))

    merged   = inferred.merge(ground, on='player_id', how='inner')
    valid    = merged[merged['inferred'].isin(['Right', 'Left'])].copy()
    correct  = (valid['inferred'] == valid['ground_truth']).sum()
    accuracy = correct / len(valid) if len(valid) > 0 else 0

    print(f"\n── Foot Inference Validation ──")
    print(f"Players compared : {len(valid)}")
    print(f"Accuracy         : {accuracy:.1%}")

    mismatches = valid[valid['inferred'] != valid['ground_truth']]
    if len(mismatches):
        print(f"\nMismatches ({len(mismatches)} players):")
        print(mismatches[['player', 'inferred', 'ground_truth']].to_string(index=False))
    else:
        print("\nNo mismatches — inference was perfect on this sample.")

    return accuracy

# Usage (after running one of the approaches above):
# accuracy = validate_inference(passes, foot_df)

## STEP 3 — Merge ground-truth foot back into passes and re-run model


In [ ]:
def merge_real_foot(passes_df, foot_df, foot_col='preferred_foot'):
    """
    Replace usage-inferred weak_foot with ground-truth preferred foot.
    foot_df must have: player_id, preferred_foot ('Right' / 'Left' / 'Both')
    After calling this, re-run Cell 5 onwards in your main notebook.
    """
    drop_cols    = ['preferred_foot', 'total', 'weak_foot']
    passes_clean = passes_df.drop(columns=[c for c in drop_cols if c in passes_df.columns])

    passes_clean = passes_clean.merge(
        foot_df[['player_id', foot_col]].rename(columns={foot_col: 'preferred_foot'}),
        on='player_id', how='left'
    )

    def flag_weak_real(row):
        if row['pass_body_part'] not in ('Right Foot', 'Left Foot'):
            return np.nan
        if row.get('preferred_foot') not in ('Right', 'Left'):
            return np.nan
        used = 'Right' if row['pass_body_part'] == 'Right Foot' else 'Left'
        return int(used != row['preferred_foot'])

    passes_clean['weak_foot'] = passes_clean.apply(flag_weak_real, axis=1)

    # Coverage report
    total   = passes_clean['pass_body_part'].isin(['Right Foot', 'Left Foot']).sum()
    covered = (passes_clean[passes_clean['pass_body_part']
               .isin(['Right Foot','Left Foot'])]['preferred_foot'].notna().sum())

    print(f"\nGround-truth coverage: {covered:,}/{total:,} foot passes ({covered/total:.1%})")
    print(f"Weak foot distribution: {passes_clean['weak_foot'].value_counts(dropna=False).to_dict()}")
    print("\nNow re-run Cell 5 onwards in laliga_360_xpass.ipynb")
    return passes_clean

# ── Full workflow example ──
# Choose ONE of the approaches above, then:
#
# Option A (FBref):
#   passes = merge_real_foot(passes, foot_fbref)
#
# Option B (Transfermarkt):
#   foot_tm = scrape_transfermarkt(players_df)
#   passes  = merge_real_foot(passes, foot_tm)
#
# Option C (FIFA 21 — recommended):
#   fifa_df = load_fifa_foot('players_21.csv')
#   foot_df = match_fifa_to_statsbomb(players_df, fifa_df)
#   passes  = merge_real_foot(passes, foot_df)
#
# Option D (Wikidata):
#   wiki_df   = fetch_wikidata_foot()
#   foot_wiki = match_wikidata_to_statsbomb(players_df, wiki_df)
#   passes    = merge_real_foot(passes, foot_wiki)

## What to expect — results comparison

## What to Expect After Switching to Ground-Truth Foot

| Metric | Inferred foot | Ground-truth foot |
|---|---|---|
| Coverage | ~80% of players | ~90-95% (FIFA/TM) |
| Weak foot accuracy | ~85-90% | 100% by definition |
| Cross miscalibration gap | −14.8pp | Likely larger (bias was understated) |
| Overall AUC change | — | Minimal (+0.001 at most) |

**Key insight:** if the inference was 85% accurate, ~15% of weak-foot passes were
labelled as strong-foot and vice versa. This dilutes the calibration signal.
The real gap will be **larger** than −14.8pp, strengthening your finding.

**Validation accuracy is itself a contribution:** reporting "our usage-based inference
was X% accurate against FIFA ground truth" is a methodological result that
tells future researchers how much they can trust this shortcut when explicit
foot data is unavailable.
